# Student Performance Analysis Using Data Analytics

**Student:** Rajeshwari  
**Project:** Student Performance Analysis  
**Domain:** Data Analytics with AI

## Objective
Analyze school student data to understand the factors affecting academic performance and build a simple AI model to predict whether a student is likely to pass or fail.

### Analysis includes
- Data loading and inspection
- Data cleaning
- Exploratory Data Analysis (EDA)
- Statistical analysis
- Data visualization
- Correlation analysis
- Pass/Fail prediction using Logistic Regression
- Model evaluation
- Key findings and conclusion


In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)


## 1. Create / Load the Dataset

For this academic project, a small realistic school dataset is generated inside the notebook. This keeps the project self-contained, so no separate dataset file is required for submission.

In [ ]:
# Generate a reproducible school student dataset
np.random.seed(42)

n = 60
student_id = np.arange(1, n + 1)
gender = np.random.choice(["Male", "Female"], n)
age = np.random.randint(14, 18, n)
class_name = np.random.choice(["9", "10", "11", "12"], n)
attendance = np.clip(np.random.normal(82, 9, n), 55, 100).round(1)
study_hours = np.clip(np.random.normal(3.5, 1.4, n), 0.5, 8).round(1)
assignment_score = np.clip(50 + attendance * 0.25 + study_hours * 3 + np.random.normal(0, 7, n), 35, 100).round(1)
previous_score = np.clip(45 + attendance * 0.25 + study_hours * 4 + np.random.normal(0, 8, n), 30, 100).round(1)

final_score = np.clip(
    0.35 * previous_score +
    0.25 * assignment_score +
    0.20 * attendance +
    3.5 * study_hours +
    np.random.normal(0, 5, n),
    20, 100
).round(1)

result = np.where(final_score >= 40, "Pass", "Fail")

df = pd.DataFrame({
    "Student_ID": student_id,
    "Gender": gender,
    "Age": age,
    "Class": class_name,
    "Attendance": attendance,
    "Study_Hours": study_hours,
    "Assignment_Score": assignment_score,
    "Previous_Score": previous_score,
    "Final_Score": final_score,
    "Result": result
})

# Introduce a few missing values for demonstration of data cleaning
df.loc[[5, 22], "Attendance"] = np.nan
df.loc[[11], "Study_Hours"] = np.nan
df.loc[[35], "Assignment_Score"] = np.nan

print("Dataset created successfully.")
df.head()


In [ ]:
# Basic dataset information
print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())


## 2. Data Cleaning

Missing numerical values are replaced using the median of their respective columns.

In [ ]:
# Fill missing numerical values with median
numeric_cols = ["Attendance", "Study_Hours", "Assignment_Score"]

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

print("Missing values after cleaning:")
print(df.isnull().sum())


In [ ]:
# Descriptive statistics
df.describe()


## 3. Exploratory Data Analysis

In [ ]:
# Pass vs Fail count
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Result")
plt.title("Pass vs Fail")
plt.xlabel("Result")
plt.ylabel("Number of Students")
plt.show()


In [ ]:
# Final score distribution
plt.figure(figsize=(7, 4))
sns.histplot(df["Final_Score"], bins=10, kde=True)
plt.title("Distribution of Final Scores")
plt.xlabel("Final Score")
plt.ylabel("Number of Students")
plt.show()


In [ ]:
# Average final score by class
class_average = df.groupby("Class")["Final_Score"].mean().sort_index()

plt.figure(figsize=(7, 4))
class_average.plot(kind="bar")
plt.title("Average Final Score by Class")
plt.xlabel("Class")
plt.ylabel("Average Final Score")
plt.xticks(rotation=0)
plt.show()

print(class_average.round(2))


In [ ]:
# Attendance vs final score
plt.figure(figsize=(7, 4))
sns.scatterplot(data=df, x="Attendance", y="Final_Score", hue="Result")
plt.title("Attendance vs Final Score")
plt.xlabel("Attendance (%)")
plt.ylabel("Final Score")
plt.show()


In [ ]:
# Study hours vs final score
plt.figure(figsize=(7, 4))
sns.scatterplot(data=df, x="Study_Hours", y="Final_Score", hue="Result")
plt.title("Study Hours vs Final Score")
plt.xlabel("Study Hours per Day")
plt.ylabel("Final Score")
plt.show()


In [ ]:
# Boxplot of final scores by result
plt.figure(figsize=(6, 4))
sns.boxplot(data=df, x="Result", y="Final_Score")
plt.title("Final Score by Result")
plt.show()


## 4. Correlation Analysis

In [ ]:
# Correlation matrix for numerical variables
corr_cols = [
    "Age", "Attendance", "Study_Hours",
    "Assignment_Score", "Previous_Score", "Final_Score"
]

plt.figure(figsize=(8, 6))
sns.heatmap(df[corr_cols].corr(), annot=True, fmt=".2f", cmap="Blues")
plt.title("Correlation Matrix")
plt.show()


## 5. Simple AI Model: Pass/Fail Prediction

A Logistic Regression model is used to predict the student's result from attendance, study hours, assignment score and previous score.

In [ ]:
# Prepare data for machine learning
features = ["Attendance", "Study_Hours", "Assignment_Score", "Previous_Score"]
X = df[features]
y = (df["Result"] == "Pass").astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.2%}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Fail", "Pass"]))


In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Fail", "Pass"],
    yticklabels=["Fail", "Pass"]
)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


In [ ]:
# Example prediction for a new student
new_student = pd.DataFrame({
    "Attendance": [88],
    "Study_Hours": [4.5],
    "Assignment_Score": [78],
    "Previous_Score": [74]
})

new_student_scaled = scaler.transform(new_student)
prediction = model.predict(new_student_scaled)[0]
probability = model.predict_proba(new_student_scaled)[0][1]

print("Predicted Result:", "Pass" if prediction == 1 else "Fail")
print(f"Probability of Pass: {probability:.2%}")


## 6. Key Findings

The analysis can be used to identify:
- Overall pass/fail distribution.
- Average performance across classes.
- Relationship between attendance and final marks.
- Relationship between study hours and final marks.
- Relationship between previous/assignment performance and final marks.
- The effectiveness of a simple Logistic Regression model for Pass/Fail prediction.

## 7. Conclusion

Student performance data can be analyzed using basic data analytics techniques to identify important academic patterns. Attendance, study habits, assignment performance and previous marks provide useful information for understanding student outcomes. A simple Logistic Regression model can also provide an introductory AI-based Pass/Fail prediction system.

This project is intended for academic demonstration and should not be used as the sole basis for real educational decisions.
